<a href="https://colab.research.google.com/github/madalamanikanta/ImageCaptioning_MiniProject/blob/main/04_CLIP_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 — Connect Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — GPU verification

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


## Step 3 : Define Project Paths


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject"
)

PROCESSED_DIR = PROJECT_DIR / "Processed"
FEATURES_DIR = PROJECT_DIR / "Features"

cleaned_file = PROCESSED_DIR / "cleaned_dataset.csv"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project Directory :", PROJECT_DIR)
print("Processed Directory:", PROCESSED_DIR)
print("Features Directory :", FEATURES_DIR)
print("Cleaned Dataset    :", cleaned_file)

Project Directory : /content/drive/MyDrive/ImageCaptioning_MiniProject
Processed Directory: /content/drive/MyDrive/ImageCaptioning_MiniProject/Processed
Features Directory : /content/drive/MyDrive/ImageCaptioning_MiniProject/Features
Cleaned Dataset    : /content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/cleaned_dataset.csv


## Step 4 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from PIL import Image
from tqdm.auto import tqdm
import torch
print("Libraries imported successfully.")

Libraries imported successfully.


## Step 5 : Load Cleaned Dataset

In [ ]:
df = pd.read_csv(cleaned_file)

In [ ]:
print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (599278, 3)


In [ ]:
df.head()

,image_path,caption,dataset
0,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A child in a pink dress is climbing up...,Flickr8K
1,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A girl going into a wooden building <end>,Flickr8K
2,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing into a wooden p...,Flickr8K
3,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing the stairs to h...,Flickr8K
4,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl in a pink dress going in...,Flickr8K


## Step 6 : Identify Unique Images


In [ ]:
unique_images = (
    df["image_path"]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
print("Total caption rows :", len(df))
print("Unique images      :", len(unique_images))

Total caption rows : 599278
Unique images      : 119865


## Step 7 : Load CLIP


In [ ]:
!pip install -q transformers

In [ ]:
from transformers import CLIPProcessor, CLIPModel

print("Transformers imported successfully.")

Transformers imported successfully.


## Step 8 — Load CLIP ViT-B/32

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print("CLIP model loaded successfully!")

Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model loaded successfully!


## Step 9 : Test CLIP Feature Extraction


In [ ]:
test_image_path = unique_images.iloc[0]

In [ ]:
print("Test image:")
print(test_image_path)

Test image:
/content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg


In [ ]:
image = Image.open(test_image_path).convert("RGB")
print("Image size:", image.size)

Image size: (375, 500)


In [ ]:
inputs = processor(
    images=image,
    return_tensors="pt"
)
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

In [ ]:
with torch.no_grad():
    image_features = model.get_image_features(**inputs)

print("Feature shape:", image_features.pooler_output.shape)

Feature shape: torch.Size([1, 512])


# Step 10 — Prepare Batch Feature Extraction


In [ ]:
import os
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# Number of images processed in one checkpoint
CHUNK_SIZE = 1000
# Directory where CLIP feature chunks will be saved
CLIP_DIR = FEATURES_DIR / "CLIP"
CLIP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print("CLIP feature directory:")
print(CLIP_DIR)

CLIP feature directory:
/content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP


In [ ]:
print("Total unique images:", len(unique_images))
print("Chunk size:", CHUNK_SIZE)

Total unique images: 119865
Chunk size: 1000


# Step 11 — Check Existing Feature Chunks

In [ ]:
existing_chunks = sorted(CLIP_DIR.glob("features_*.npy"))
print("Existing feature chunks:", len(existing_chunks))
if existing_chunks:
    print("Last existing chunk:")
    print(existing_chunks[-1])
else:
    print("No previous feature chunks found.")

Existing feature chunks: 0
No previous feature chunks found.


## Step 12 — Batch CLIP Feature Extraction

In [ ]:
# Step 12 — Batch CLIP Feature Extraction

failed_images = []
total_images = len(unique_images)
for start in range(0, total_images, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, total_images)
    chunk_number = start // CHUNK_SIZE
    output_file = CLIP_DIR / f"features_{chunk_number:04d}.npy"
    # Skip this chunk if it was already processed
    if output_file.exists():
        print(f"Skipping chunk {chunk_number}: already exists.")
        continue
    print()
    print("=" * 60)
    print(f"Processing images {start} → {end - 1}")
    print(f"Chunk {chunk_number}")
    print("=" * 60)
    batch_features = []
    for image_path in tqdm(
        unique_images.iloc[start:end],
        desc=f"Chunk {chunk_number}"
    ):
        try:
            image = Image.open(image_path).convert("RGB")
            inputs = processor(
                images=image,
                return_tensors="pt"
            )
            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }
            with torch.no_grad():
                output = model.get_image_features(**inputs)
            # Transformers version returns an output object
            feature = output.pooler_output
            # Convert to NumPy
            feature = feature.cpu().numpy()
            batch_features.append(feature[0])
        except Exception as e:
            print(f"Failed: {image_path}")
            print("Error:", e)
            failed_images.append({
                "image_path": str(image_path),
                "error": str(e)
            })
            # Keep alignment by inserting zeros
            batch_features.append(
                np.zeros(512, dtype=np.float32)
            )
    # Convert chunk to NumPy array
    batch_features = np.array(
        batch_features,
        dtype=np.float32
    )
    # Save chunk to Google Drive
    np.save(output_file, batch_features)
    print()
    print(f"Saved: {output_file}")
    print(f"Feature shape: {batch_features.shape}")


Processing images 0 → 999
Chunk 0


Chunk 0:   0%|          | 0/1000 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0000.npy
Feature shape: (1000, 512)

Processing images 1000 → 1999
Chunk 1


Chunk 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0001.npy
Feature shape: (1000, 512)

Processing images 2000 → 2999
Chunk 2


...

Chunk 71:   0%|          | 0/1000 [00:00<?, ?it/s]

Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000027764.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000158080.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000127100.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000065924.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000313709.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000319938.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning

Chunk 72:   0%|          | 0/1000 [00:00<?, ?it/s]

Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000174718.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000110371.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000256720.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000162712.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000119292.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000012728.jpg
Error: [Errno 5] Input/output error
Failed: /content/drive/MyDrive/ImageCaptioning

OSError: [Errno 107] Transport endpoint is not connected: '/content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0072.npy'

## Rough Work

In [ ]:
from pathlib import Path

FEATURE_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP"
)

files = sorted(FEATURE_DIR.glob("features_*.npy"))

print("Feature files found:", len(files))

if files:
    numbers = sorted(int(f.stem.split("_")[1]) for f in files)

    print("First chunk:", numbers[0])
    print("Last chunk:", numbers[-1])

    missing = [
        i for i in range(numbers[-1] + 1)
        if i not in numbers
    ]

    print("Missing chunks:", missing[:20])
    print("Total missing:", len(missing))

Feature files found: 71
First chunk: 0
Last chunk: 70
Missing chunks: []
Total missing: 0


In [ ]:
# Step 12A — Check Existing CLIP Features and Unique Images

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject"
)

PROCESSED_DIR = PROJECT_DIR / "Processed"
FEATURE_DIR = PROJECT_DIR / "Features" / "CLIP"

# Load cleaned dataset
cleaned_file = PROCESSED_DIR / "cleaned_dataset.csv"
df = pd.read_csv(cleaned_file)

# Unique images
unique_images = df["image_path"].drop_duplicates().reset_index(drop=True)

# Existing feature files
feature_files = sorted(FEATURE_DIR.glob("features_*.npy"))

print("=" * 60)
print("CLIP EXTRACTION STATUS")
print("=" * 60)

print("Total caption rows :", len(df))
print("Unique images      :", len(unique_images))
print("Feature files      :", len(feature_files))

if feature_files:
    numbers = sorted(
        int(f.stem.split("_")[1])
        for f in feature_files
    )

    print("Existing chunks    :", numbers[0], "->", numbers[-1])
    print("Existing images    :", len(feature_files) * 1000)

print("=" * 60)

CLIP EXTRACTION STATUS
Total caption rows : 599278
Unique images      : 119865
Feature files      : 71
Existing chunks    : 0 -> 70
Existing images    : 71000


##Step 12 — Resume CLIP Feature Extraction

In [ ]:
# ============================================================
# Step 12 — Optimized CLIP Feature Extraction
# Resume from Chunk 74 → 119
# ============================================================

import time
import numpy as np
import torch

from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor


# ============================================================
# 1. DEVICE SETUP
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("OPTIMIZED CLIP FEATURE EXTRACTION")
print("=" * 60)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is NOT available.")


# ============================================================
# 2. CONFIGURATION
# ============================================================

START_CHUNK = 74
END_CHUNK = 119

CHUNK_SIZE = 1000

# Number of images sent to CLIP at once
BATCH_SIZE = 64

# Number of threads used for loading images
NUM_WORKERS = 8

# Retry temporary Google Drive I/O errors
MAX_RETRIES = 5


print()
print("Total unique images :", len(unique_images))
print("Chunks              :", START_CHUNK, "→", END_CHUNK)
print("Batch size          :", BATCH_SIZE)
print("Image workers       :", NUM_WORKERS)


# ============================================================
# 3. VERIFY CLIP MODEL DEVICE
# ============================================================

model = model.to(device)
model.eval()

print()
print("CLIP model is ready.")
print("Model device:", next(model.parameters()).device)


# ============================================================
# 4. IMAGE LOADING FUNCTION
# ============================================================

def load_image_with_retry(image_path):

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            with Image.open(image_path) as img:

                image = img.convert("RGB").copy()

            return image, None

        except Exception as e:

            if attempt < MAX_RETRIES:

                time.sleep(1)

            else:

                return None, {
                    "image_path": str(image_path),
                    "error": str(e)
                }


# ============================================================
# 5. PROCESS CHUNKS
# ============================================================

failed_images = []

for chunk_number in range(
    START_CHUNK,
    END_CHUNK + 1
):

    start = chunk_number * CHUNK_SIZE

    end = min(
        start + CHUNK_SIZE,
        len(unique_images)
    )

    output_file = (
        CLIP_DIR /
        f"features_{chunk_number:04d}.npy"
    )


    # --------------------------------------------------------
    # Skip already completed chunks
    # --------------------------------------------------------

    if output_file.exists():

        print(
            f"Skipping chunk {chunk_number} "
            f"({start} → {end - 1}) — already exists."
        )

        continue


    print()
    print("=" * 60)
    print(f"Processing chunk {chunk_number}")
    print(f"Images {start} → {end - 1}")
    print("=" * 60)


    chunk_paths = (
        unique_images
        .iloc[start:end]
        .tolist()
    )


    chunk_features = []

    chunk_start_time = time.time()


    # ========================================================
    # 6. PROCESS CHUNK IN GPU BATCHES
    # ========================================================

    for batch_start in tqdm(
        range(
            0,
            len(chunk_paths),
            BATCH_SIZE
        ),
        desc=f"Chunk {chunk_number}",
        unit="batch"
    ):

        batch_paths = chunk_paths[
            batch_start:
            batch_start + BATCH_SIZE
        ]


        # ----------------------------------------------------
        # Load images in parallel
        # ----------------------------------------------------

        with ThreadPoolExecutor(
            max_workers=NUM_WORKERS
        ) as executor:

            results = list(
                executor.map(
                    load_image_with_retry,
                    batch_paths
                )
            )


        images = []
        failed_batch = []


        for image, error in results:

            if image is not None:

                images.append(image)

            else:

                failed_batch.append(error)


        # ----------------------------------------------------
        # IMPORTANT:
        # Never silently create fake features.
        #
        # If even one image fails, stop before saving the
        # chunk so image-feature alignment remains correct.
        # ----------------------------------------------------

        if failed_batch:

            failed_images.extend(
                failed_batch
            )

            failed_paths = [
                item["image_path"]
                for item in failed_batch
            ]

            raise RuntimeError(
                "\nImage loading failed.\n"
                "The current chunk was NOT saved.\n\n"
                f"Failed images:\n{failed_paths[:10]}\n\n"
                "Run Step 12 again to retry."
            )


        # ====================================================
        # 7. CLIP PREPROCESSING
        # ====================================================

        inputs = processor(
            images=images,
            return_tensors="pt"
        )


        # Move tensors to T4 GPU

        inputs = {
            key: value.to(
                device,
                non_blocking=True
            )
            for key, value in inputs.items()
        }


        # ====================================================
        # 8. CLIP FEATURE EXTRACTION
        # ====================================================

        with torch.inference_mode():

            if device.type == "cuda":

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    output = model.get_image_features(
                        **inputs
                    )

            else:

                output = model.get_image_features(
                    **inputs
                )


        # ====================================================
        # 9. GET 512-D CLIP FEATURES
        # ====================================================

        if hasattr(
            output,
            "pooler_output"
        ):

            features = output.pooler_output

        else:

            features = output


        # ====================================================
        # 10. NORMALIZE FEATURES
        # ====================================================

        features = features / features.norm(
            dim=-1,
            keepdim=True
        )


        # Move from GPU → CPU

        features = (
            features
            .detach()
            .cpu()
            .float()
            .numpy()
        )


        # Add features from this batch

        chunk_features.append(
            features
        )


        # Free temporary objects

        del inputs
        del images
        del output
        del features


    # ========================================================
    # 11. COMBINE BATCHES
    # ========================================================

    chunk_features = np.concatenate(
        chunk_features,
        axis=0
    )


    # ========================================================
    # 12. SAFETY CHECK
    # ========================================================

    expected_images = len(chunk_paths)

    print()
    print(
        "Expected features:",
        expected_images
    )

    print(
        "Actual features:",
        len(chunk_features)
    )


    if chunk_features.shape != (
        expected_images,
        512
    ):

        raise RuntimeError(
            f"\nFeature shape mismatch!\n"
            f"Expected: ({expected_images}, 512)\n"
            f"Got: {chunk_features.shape}\n\n"
            "Chunk was NOT saved."
        )


    # ========================================================
    # 13. SAVE FEATURES
    # ========================================================

    np.save(
        output_file,
        chunk_features
    )


    elapsed = (
        time.time()
        - chunk_start_time
    )


    print()
    print(
        f"Saved: {output_file}"
    )

    print(
        "Feature shape:",
        chunk_features.shape
    )

    print(
        f"Time: {elapsed / 60:.2f} minutes"
    )


# ============================================================
# 14. FINAL STATUS
# ============================================================

print()
print("=" * 60)
print("CLIP EXTRACTION FINISHED")
print("=" * 60)

feature_files = sorted(
    CLIP_DIR.glob("features_*.npy")
)

print(
    "Feature files found:",
    len(feature_files)
)

print(
    "Failed images recorded:",
    len(failed_images)
)

if feature_files:

    print(
        "First:",
        feature_files[0].name
    )

    print(
        "Last:",
        feature_files[-1].name
    )

OPTIMIZED CLIP FEATURE EXTRACTION
Device: cuda
GPU: Tesla T4

Total unique images : 119865
Chunks              : 74 → 119
Batch size          : 64
Image workers       : 8

CLIP model is ready.
Model device: cuda:0

Processing chunk 74
Images 74000 → 74999


Chunk 74:   0%|          | 0/16 [00:00<?, ?batch/s]


Expected features: 1000
Actual features: 1000

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0074.npy
Feature shape: (1000, 512)
Time: 1.03 minutes

Processing chunk 75
Images 75000 → 75999


Chunk 75:   0%|          | 0/16 [00:00<?, ?batch/s]


Expected features: 1000
Actual features: 1000

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0075.npy
Feature shape: (1000, 512)
Time: 1.78 minutes

Processing chunk 76
Images 76000 → 76999


...


Expected features: 1000
Actual features: 1000

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0117.npy
Feature shape: (1000, 512)
Time: 1.79 minutes

Processing chunk 118
Images 118000 → 118999


Chunk 118:   0%|          | 0/16 [00:00<?, ?batch/s]


Expected features: 1000
Actual features: 1000

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0118.npy
Feature shape: (1000, 512)
Time: 1.77 minutes

Processing chunk 119
Images 119000 → 119864


Chunk 119:   0%|          | 0/14 [00:00<?, ?batch/s]


Expected features: 865
Actual features: 865

Saved: /content/drive/MyDrive/ImageCaptioning_MiniProject/Features/CLIP/features_0119.npy
Feature shape: (865, 512)
Time: 1.54 minutes

CLIP EXTRACTION FINISHED
Feature files found: 120
Failed images recorded: 0
First: features_0000.npy
Last: features_0119.npy
